# Apache Spark — The DataFrame API (Complete Guide)

This notebook collects **everything about Spark DataFrames** from the Day 2 – Day 4 lectures and labs into one place, in a logical teaching order.
Every section explains *what* a feature is, *why* it exists, and then shows runnable code.

| # | Section | What you will learn |
|---|---------|---------------------|
| | **Part 1 — Getting started** | |
| 0 | Setup | Start Spark and point it to the data folder |
| 1 | Why DataFrames? | The same problem solved with RDDs and with DataFrames |
| 2 | Core concepts | Immutability, lazy evaluation, transformations vs actions |
| | **Part 2 — Creating & understanding a DataFrame** | |
| 3 | Creating DataFrames | From Python lists and from JSON files, working with `Row` objects |
| 4 | Data types & schemas | Spark types, DDL vs `StructType`, schema enforcement |
| 5 | Inspecting a DataFrame | `printSchema`, `describe`, `summary`, `columns`, `dtypes` |
| | **Part 3 — Transforming data** | |
| 6 | Columns & `select` | `df.col`, `df['col']`, `col()`, `expr()`, `alias()` |
| 7 | Filtering rows | `filter` / `where`, string vs column conditions |
| 8 | Adding, renaming, dropping columns | `withColumn`, `concat_ws`, `withColumnRenamed`, `drop` |
| 9 | Sorting | `sort` / `orderBy`, ascending & descending |
| 10 | Grouping & aggregation | `groupBy`, `agg`, `avg`, `sum`, `count` |
| 11 | Conditional columns | `when` / `otherwise` |
| 12 | User-Defined Functions | `udf`, the `@udf` decorator, performance notes |
| | **Part 4 — Reading & writing data** | |
| 13 | Reading CSV files correctly | `header`, `inferSchema`, `schema`, `samplingRatio`, read modes |
| 14 | Missing data | `na.drop` and `na.fill` |
| 15 | Data sources | `DataFrameReader`, `DataFrameWriter`, Parquet / CSV / JSON / JDBC |
| | **Part 5 — Performance & practice** | |
| 16 | Caching | `cache`, `unpersist`, `clearCache` and how to measure the effect |
| 17 | Case study: SF Fire Calls | Dates, casting, distinct, real aggregation questions — everything together |
| | **Part 6 — Streaming** | |
| 18 | Structured Streaming | Streaming DataFrames, sinks, output modes, triggers, checkpoints |
| 19 | 📋 Cheat sheet | A summary of the whole notebook |

> The companion notebook **`Spark_SQL.ipynb`** covers Spark SQL: temp views, databases, managed/unmanaged tables and SQL queries.

---
## 0. Setup

### 0.1 Start a SparkSession
The **SparkSession** is the single entry point to all Spark functionality (DataFrames, SQL, streaming).
There can only be **one** SparkSession per JVM — `getOrCreate()` returns the existing one if it is already running.

The older entry point, **SparkContext** (`sc`), is still available inside the session and is what RDDs use.

In [1]:
import warnings
warnings.filterwarnings('ignore')

from pyspark.sql import SparkSession
import pyspark.sql.functions as F          # all built-in column functions: F.col, F.avg, F.sum, ...
from pyspark.sql.types import *            # data types: StringType, IntegerType, StructType, ...

# Stop Jupyter from wrapping wide tables printed by df.show()
from IPython.display import display, HTML
display(HTML("<style>pre { white-space: pre !important; } </style>"))

spark = (SparkSession
         .builder
         .appName("DataFrames")
         .getOrCreate())

sc = spark.sparkContext   # the lower-level entry point (used for RDDs)
sc.setLogLevel("ERROR")   # hide Spark's INFO/WARN log lines so cell outputs stay readable
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/30 05:14:16 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/10/30 05:14:17 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


### 0.2 Where is the data?
Spark runs on a **YARN cluster**: the driver runs here in Jupyter, but the **executors run on the worker machines**.
A file on the Jupyter machine's local disk is invisible to the executors, so all the data lives in **HDFS**, which every machine can read.

- `DATA_DIR` — the course input files (`departuredelays.csv`, `sf-fire-calls.csv`, `NullData.csv`, `people.json`, and the `csv/`, `json/`, `parquet/` folders).
- `OUT_DIR` — where this notebook writes its output files.

> Upload the data once from a terminal: `hdfs dfs -put <local DataFiles folder> /user/user/data/spark_course`

In [2]:
# A path without a scheme uses the cluster's default filesystem (fs.defaultFS = hdfs://bdss),
# so "/user/user/..." means "hdfs://bdss/user/user/...".
# Avoid "hdfs:///..." (no host): batch reads accept it, but streaming checkpoints fail with "Uri without authority".
DATA_DIR = "/user/user/data/spark_course"
OUT_DIR  = "/user/user/data/spark_course_output"

> **Why `import pyspark.sql.functions as F` and not `from pyspark.sql.functions import *`?**
> The star import silently **replaces Python's built-in** `sum`, `min`, `max`, `round`, `abs`, ... with Spark's versions.
> After that, a normal line like `sum([1, 2, 3])` breaks. Using the `F.` prefix keeps the two worlds separate and makes it obvious which functions belong to Spark.

---
## 1. Why DataFrames? — The same problem with RDDs and with DataFrames

**Task:** we have `(name, age)` pairs. Group them by name and compute the **average age** of each name.

### 1.1 The RDD way
An RDD is just a distributed collection of Python objects — Spark knows nothing about what is inside them.

In [3]:
dataRDD = sc.parallelize([("Brooke", 20), ("Denny", 31), ("Jules", 30),
                          ("TD", 35), ("Brooke", 25), ("Jules", 40), ("Denny", 51)])

print(dataRDD.first())     # first element
print(dataRDD.take(2))     # first 2 elements
print(dataRDD.collect())   # ALL elements, brought back to the driver

('Brooke', 20)
[('Brooke', 20), ('Denny', 31)]
[('Brooke', 20), ('Denny', 31), ('Jules', 30), ('TD', 35), ('Brooke', 25), ('Jules', 40), ('Denny', 51)]


**Option A — `groupByKey`:** collect all ages of a name together, then compute `sum / len`.

In [4]:
# x looks like ('Brooke', <iterable of ages>)
rdd_avg = dataRDD.groupByKey().map(lambda x: (x[0], sum(x[1]) / len(x[1])))
rdd_avg.collect()

[('Brooke', 22.5), ('Denny', 41.0), ('TD', 35.0), ('Jules', 35.0)]

**Option B — `map` + `reduceByKey`** (more efficient, because values are combined *before* the shuffle). Let's follow it step by step:

| Step | Code | One element looks like |
|------|------|------------------------|
| 1 | `map(lambda x: (x[0], (x[1], 1)))` | `('Brooke', (20, 1))` → (name, (age, count)) |
| 2 | `reduceByKey(lambda x, y: (x[0]+y[0], x[1]+y[1]))` | `('Brooke', (45, 2))` → (name, (total age, total count)) |
| 3 | `map(lambda x: (x[0], x[1][0] / x[1][1]))` | `('Brooke', 22.5)` → (name, average) |

In [5]:
s1 = dataRDD.map(lambda x: (x[0], (x[1], 1)))
print("Step 1:", s1.take(2))

s2 = s1.reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1]))
print("Step 2:", s2.collect())

s3 = s2.map(lambda x: (x[0], x[1][0] / x[1][1]))
print("Step 3:", s3.collect())

Step 1: [('Brooke', (20, 1)), ('Denny', (31, 1))]
Step 2: [('Brooke', (45, 2)), ('Denny', (82, 2)), ('Jules', (70, 2)), ('TD', (35, 1))]
Step 3: [('Brooke', 22.5), ('Denny', 41.0), ('TD', 35.0), ('Jules', 35.0)]


In [6]:
# The same three steps chained together
agesRDD = (dataRDD
           .map(lambda x: (x[0], (x[1], 1)))
           .reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1]))
           .map(lambda x: (x[0], x[1][0] / x[1][1])))
agesRDD.collect()

[('Brooke', 22.5), ('Denny', 41.0), ('Jules', 35.0), ('TD', 35.0)]

This works, but it is **cryptic**: the code tells Spark **how** to compute the result with a chain of lambdas.
Spark cannot see *inside* the lambdas, so it cannot understand the intention or optimize anything.

### 1.2 The DataFrame way
Now we tell Spark **what** we want, using high-level operators:

In [7]:
data_df = spark.createDataFrame([("Brooke", 20), ("Denny", 31), ("Jules", 30),
                                 ("TD", 35), ("Brooke", 25), ("Jules", 40), ("Denny", 51)],
                                ['name', 'age'])

avg_df = data_df.groupBy('name').agg(F.avg('age').alias('Average'))
avg_df.show()

+------+-------+
|  name|Average|
+------+-------+
|Brooke|   22.5|
| Jules|   35.0|
| Denny|   41.0|
|    TD|   35.0|
+------+-------+



One readable line. Because Spark can **parse and understand** this query (group by `name`, average of `age`), its optimizer (**Catalyst**) can rearrange and optimize the operations for efficient execution.
This is the main reason the DataFrame API is preferred over raw RDDs for structured data.

---
## 2. Core concepts

### 2.1 What is a DataFrame?
- Inspired by **pandas DataFrames**: a Spark DataFrame is like a **distributed in-memory table** with **named columns** and a **schema**, where every column has a specific data type (integer, string, array, map, date, timestamp, ...).
- It is split into **partitions** that live on different machines (executors) of the cluster.
- DataFrames can be built from many sources: structured files (CSV, JSON, Parquet), Hive tables, external databases (JDBC) or existing RDDs.

### 2.2 Dataset vs DataFrame
- A **Dataset** is a distributed collection of *typed* objects. It combines RDD benefits (strong typing, lambdas) with Spark SQL's optimized engine. It exists only in **Scala and Java**.
- A **DataFrame** is a Dataset organized into **named columns** — conceptually a table in a relational database.
- Python has no Dataset API (Python is dynamically typed), but you already get most of the benefits: e.g. you can read a field by name with `row.columnName`.
- The DataFrame API is available in **Scala, Java, Python and R**.

### 2.3 DataFrames are immutable
You never change a DataFrame in place. Every operation (`select`, `withColumn`, `filter`, ...) returns a **new** DataFrame, and Spark keeps the **lineage** (the recipe of transformations) to rebuild any result if a machine fails.

### 2.4 Transformations vs actions (lazy evaluation)
| Type | What it does | Examples |
|------|--------------|----------|
| **Transformation** | Describes a new DataFrame. **Nothing runs yet** (lazy). | `select`, `filter`, `where`, `withColumn`, `groupBy`, `sort`, `join`, `distinct` |
| **Action** | Triggers the actual computation and returns a result to the driver or writes data. | `show`, `count`, `collect`, `take`, `head`, `first`, `write...save` |

Being lazy lets Spark look at the **whole plan** before running it, and optimize it as one unit.

### 2.5 Narrow vs wide transformations
- **Narrow**: each input partition contributes to only **one** output partition (e.g. `select`, `filter`, `withColumn`). Fast, no data movement.
- **Wide**: input partitions contribute to **many** output partitions (e.g. `groupBy`, `distinct`, `sort`, `join`). Spark must **shuffle** data across the cluster, which is expensive.

In [8]:
# Transformation: returns immediately, nothing is computed yet
doubled = data_df.withColumn('age_x2', F.col('age') * 2)
print(doubled)        # just a description: DataFrame[name: string, age: bigint, age_x2: bigint]

# Action: now Spark actually runs a job
doubled.show()

DataFrame[name: string, age: bigint, age_x2: bigint]
+------+---+------+
|  name|age|age_x2|
+------+---+------+
|Brooke| 20|    40|
| Denny| 31|    62|
| Jules| 30|    60|
|    TD| 35|    70|
|Brooke| 25|    50|
| Jules| 40|    80|
| Denny| 51|   102|
+------+---+------+



---
## 3. Creating DataFrames

### 3.1 From a Python list
`spark.createDataFrame(data, schema)` — the schema can be a list of column names, a DDL string or a `StructType` (see section 4).

In [9]:
data_df.show()
data_df.printSchema()
print("Number of partitions:", data_df.rdd.getNumPartitions())

+------+---+
|  name|age|
+------+---+
|Brooke| 20|
| Denny| 31|
| Jules| 30|
|    TD| 35|
|Brooke| 25|
| Jules| 40|
| Denny| 51|
+------+---+

root
 |-- name: string (nullable = true)
 |-- age: long (nullable = true)

Number of partitions: 2


### 3.2 Getting rows back to Python: `collect`, `head`, `take`
These actions return **`Row`** objects. A `Row` behaves like a tuple *and* like an object with named fields.

> **Warning:** `collect()` brings **all** rows to the driver. On a big dataset this can crash the driver with an out-of-memory error. Prefer `show()`, `take(n)` or `head(n)` to look at data.

In [10]:
rows = data_df.collect()     # list of Row objects
print(rows[0])               # Row(name='Brooke', age=20)
print(rows[0][1])            # access by position -> 20
print(rows[0].name)          # access by field name -> 'Brooke'
print(rows[0]['age'])        # access like a dict -> 20
print(data_df.head(3))       # first 3 rows as a list

Row(name='Brooke', age=20)
20
Brooke
20
[Row(name='Brooke', age=20), Row(name='Denny', age=31), Row(name='Jules', age=30)]


### 3.3 From a JSON file
Spark expects **JSON Lines** by default: one complete JSON object per line.
`spark.read.text()` shows the raw lines of the file, so we can see what Spark will parse:

In [11]:
spark.read.text(f"{DATA_DIR}/people.json").show(truncate=False)     # the raw file, one line per row

df = spark.read.json(f"{DATA_DIR}/people.json")
df.show()
df.printSchema()

+---------------------------+
|value                      |
+---------------------------+
|{"name":"Michael"}         |
|{"name":"Andy", "age":30}  |
|{"name":"Justin", "age":19}|
+---------------------------+

+----+-------+
| age|   name|
+----+-------+
|NULL|Michael|
|  30|   Andy|
|  19| Justin|
+----+-------+

root
 |-- age: long (nullable = true)
 |-- name: string (nullable = true)



Notice:
- Michael has no `age`, so Spark fills it with **`null`**.
- Spark **inferred** the types by reading the file: `age` became `long` (JSON numbers are read as 64-bit integers).

---
## 4. Spark data types & schemas

### 4.1 Basic data types
| Spark type | Python type | Notes |
|-----------|-------------|-------|
| `ByteType` | `int` | 1-byte integer |
| `ShortType` | `int` | 2-byte integer |
| `IntegerType` | `int` | 4-byte integer |
| `LongType` | `int` | 8-byte integer |
| `FloatType` | `float` | 4-byte decimal number |
| `DoubleType` | `float` | 8-byte decimal number |
| `DecimalType` | `decimal.Decimal` | exact decimal (money) |
| `StringType` | `str` | text |
| `BooleanType` | `bool` | True / False |

### 4.2 Structured and complex types
| Spark type | Python type | Example |
|-----------|-------------|---------|
| `BinaryType` | `bytearray` | raw bytes |
| `DateType` | `datetime.date` | 2024-01-31 |
| `TimestampType` | `datetime.datetime` | 2024-01-31 10:30:00 |
| `ArrayType(elementType)` | `list` | `["twitter", "LinkedIn"]` |
| `MapType(keyType, valueType)` | `dict` | `{"a": 1}` |
| `StructType([...])` | list or tuple | a row inside a row (nested record) |
| `StructField(name, type, nullable)` | — | one column inside a `StructType` |

> The list of types is less important than **how they come together when you define a schema** for your data.

### 4.3 What is a schema and why define it up front?
A **schema** defines the column names and their data types. Instead of letting Spark guess (*schema-on-read*), defining it up front gives three benefits:
1. Spark does **not** have to infer the data types.
2. Spark does **not** launch a separate job just to read a large part of the file to guess the schema — for a big file that is slow and expensive.
3. You **detect errors early** when the data does not match the schema.

*So: always define your schema when reading a large file.*

### 4.4 Two ways to define a schema
**(a) Programmatically with `StructType` / `StructField`:**

In [12]:
# StructField(column name, data type, nullable?)
book_schema = StructType([StructField("author", StringType(),  False),
                          StructField("title",  StringType(),  False),
                          StructField("pages",  IntegerType(), False)])
book_schema

StructType([StructField('author', StringType(), False), StructField('title', StringType(), False), StructField('pages', IntegerType(), False)])

**(b) With a DDL string** (Data Definition Language) — much simpler to read and write:

In [13]:
schema = "Id INT NOT NULL, First STRING, Last STRING, Url STRING, Published STRING, Hits INT, Campaigns ARRAY<STRING>"

# Back-quotes (`) around names are optional for simple names, but REQUIRED when a name contains a space:
#   "`The Id` INT NOT NULL, ..."   <- correct
#   "The Id INT NOT NULL, ..."     <- wrong: Spark reads 'The' as the name and 'Id' as the type

### 4.5 Creating a DataFrame with a schema
Data: blogs written by authors and advertised on social media.

In [14]:
data = [[1, "Jules",     "Damji",   "https://tinyurl.1", "1/4/2016",  4535,  ["twitter", "LinkedIn"]],
        [2, "Brooke",    "Wenig",   "https://tinyurl.2", "5/5/2018",  8908,  ["twitter", "LinkedIn"]],
        [3, "Denny",     "Lee",     "https://tinyurl.3", "6/7/2019",  7659,  ["web", "twitter", "FB", "LinkedIn"]],
        [4, "Tathagata", "Das",     "https://tinyurl.4", "5/12/2018", 10568, ["twitter", "FB"]],
        [5, "Matei",     "Zaharia", "https://tinyurl.5", "5/14/2014", 40578, ["web", "twitter", "FB", "LinkedIn"]],
        [6, "Reynold",   "Xin",     "https://tinyurl.6", "3/2/2015",  25568, ["twitter", "LinkedIn"]]]

blogs_df = spark.createDataFrame(data, schema)
blogs_df.show()
blogs_df.printSchema()

+---+---------+-------+-----------------+---------+-----+--------------------+
| Id|    First|   Last|              Url|Published| Hits|           Campaigns|
+---+---------+-------+-----------------+---------+-----+--------------------+
|  1|    Jules|  Damji|https://tinyurl.1| 1/4/2016| 4535| [twitter, LinkedIn]|
|  2|   Brooke|  Wenig|https://tinyurl.2| 5/5/2018| 8908| [twitter, LinkedIn]|
|  3|    Denny|    Lee|https://tinyurl.3| 6/7/2019| 7659|[web, twitter, FB...|
|  4|Tathagata|    Das|https://tinyurl.4|5/12/2018|10568|       [twitter, FB]|
|  5|    Matei|Zaharia|https://tinyurl.5|5/14/2014|40578|[web, twitter, FB...|
|  6|  Reynold|    Xin|https://tinyurl.6| 3/2/2015|25568| [twitter, LinkedIn]|
+---+---------+-------+-----------------+---------+-----+--------------------+

root
 |-- Id: integer (nullable = false)
 |-- First: string (nullable = true)
 |-- Last: string (nullable = true)
 |-- Url: string (nullable = true)
 |-- Published: string (nullable = true)
 |-- Hits: integer 

Compare with creating it **without** a schema:

In [15]:
# No schema at all -> columns are named _1, _2, ... and types are guessed
spark.createDataFrame(data).printSchema()

# Only column names -> good names, but types are still guessed (Hits becomes long, not int)
spark.createDataFrame(data, ['Id', 'First', 'Last', 'Url', 'Published', 'Hits', 'Campaigns']).printSchema()

root
 |-- _1: long (nullable = true)
 |-- _2: string (nullable = true)
 |-- _3: string (nullable = true)
 |-- _4: string (nullable = true)
 |-- _5: string (nullable = true)
 |-- _6: long (nullable = true)
 |-- _7: array (nullable = true)
 |    |-- element: string (containsNull = true)

root
 |-- Id: long (nullable = true)
 |-- First: string (nullable = true)
 |-- Last: string (nullable = true)
 |-- Url: string (nullable = true)
 |-- Published: string (nullable = true)
 |-- Hits: long (nullable = true)
 |-- Campaigns: array (nullable = true)
 |    |-- element: string (containsNull = true)



### 4.6 Schema enforcement
With a schema, Spark **checks** every value. Here row 4 has `Hits = '10568'` as a **string** instead of an integer:

In [16]:
baddata = [row[:] for row in data]
baddata[3][5] = '10568'          # string instead of int

try:
    spark.createDataFrame(baddata, schema)
except Exception as e:
    print(type(e).__name__, "->", str(e)[:200])

PySparkTypeError -> [FIELD_DATA_TYPE_UNACCEPTABLE_WITH_NAME] field Hits: IntegerType() can not accept object '10568' in type <class 'str'>.


In [17]:
# verifySchema=False skips the check. The bad value is NOT converted:
# Hits for row 4 silently becomes NULL - the data is corrupted without any error. Use it only when you trust the data.
spark.createDataFrame(baddata, schema, verifySchema=False).show()

+---+---------+-------+-----------------+---------+-----+--------------------+
| Id|    First|   Last|              Url|Published| Hits|           Campaigns|
+---+---------+-------+-----------------+---------+-----+--------------------+
|  1|    Jules|  Damji|https://tinyurl.1| 1/4/2016| 4535| [twitter, LinkedIn]|
|  2|   Brooke|  Wenig|https://tinyurl.2| 5/5/2018| 8908| [twitter, LinkedIn]|
|  3|    Denny|    Lee|https://tinyurl.3| 6/7/2019| 7659|[web, twitter, FB...|
|  4|Tathagata|    Das|https://tinyurl.4|5/12/2018| NULL|       [twitter, FB]|
|  5|    Matei|Zaharia|https://tinyurl.5|5/14/2014|40578|[web, twitter, FB...|
|  6|  Reynold|    Xin|https://tinyurl.6| 3/2/2015|25568| [twitter, LinkedIn]|
+---+---------+-------+-----------------+---------+-----+--------------------+



### 4.7 Reusing a schema
Every DataFrame carries its schema. You can take it from one DataFrame and pass it when reading/creating another one.
A common trick: read a file **once** with `inferSchema=True`, copy `df.schema`, and use it from then on to skip inference.

In [18]:
print(blogs_df.schema)
print(blogs_df.columns)

saved_schema = blogs_df.schema
spark.createDataFrame(data, saved_schema).printSchema()

StructType([StructField('Id', IntegerType(), False), StructField('First', StringType(), True), StructField('Last', StringType(), True), StructField('Url', StringType(), True), StructField('Published', StringType(), True), StructField('Hits', IntegerType(), True), StructField('Campaigns', ArrayType(StringType(), True), True)])
['Id', 'First', 'Last', 'Url', 'Published', 'Hits', 'Campaigns']
root
 |-- Id: integer (nullable = false)
 |-- First: string (nullable = true)
 |-- Last: string (nullable = true)
 |-- Url: string (nullable = true)
 |-- Published: string (nullable = true)
 |-- Hits: integer (nullable = true)
 |-- Campaigns: array (nullable = true)
 |    |-- element: string (containsNull = true)



---
## 5. Inspecting a DataFrame
| Method | Returns |
|--------|---------|
| `printSchema()` | the schema as a tree |
| `columns` | list of column names |
| `dtypes` | list of `(column, type)` pairs |
| `count()` | number of rows (action) |
| `describe()` | count, mean, stddev, min, max |
| `summary()` | same as `describe()` **plus** 25%, 50% (median) and 75% percentiles |

`describe()` and `summary()` return **DataFrames**, so you need `.show()` to see them.

In [19]:
df = spark.read.json(f"{DATA_DIR}/people.json")

print(df.columns)
print(df.dtypes)
print("rows:", df.count())
df.describe().show()
df.summary().show()

['age', 'name']
[('age', 'bigint'), ('name', 'string')]
rows: 3
+-------+------------------+-------+
|summary|               age|   name|
+-------+------------------+-------+
|  count|                 2|      3|
|   mean|              24.5|   NULL|
| stddev|7.7781745930520225|   NULL|
|    min|                19|   Andy|
|    max|                30|Michael|
+-------+------------------+-------+

+-------+------------------+-------+
|summary|               age|   name|
+-------+------------------+-------+
|  count|                 2|      3|
|   mean|              24.5|   NULL|
| stddev|7.7781745930520225|   NULL|
|    min|                19|   Andy|
|    25%|                19|   NULL|
|    50%|                19|   NULL|
|    75%|                30|   NULL|
|    max|                30|Michael|
+-------+------------------+-------+



---
## 6. Columns and `select` (projection)

### 6.1 Ways to refer to a column
| Style | Example | Notes |
|-------|---------|-------|
| plain string | `'age'` | only a **name** — cannot do math on it |
| attribute | `df.age` | convenient, but breaks if the name has spaces or clashes with a DataFrame method (e.g. a column called `count`) |
| indexing | `df['age']` | **recommended**, always works |
| `col()` | `F.col('age')` | does not need the DataFrame variable; the name is only checked when the query runs |
| `expr()` | `F.expr('age + 1')` | write a small **SQL expression** as a string |

In [20]:
print(df.age)
print(df['age'])
print(F.col('age'))
print(F.col('this_column_does_not_exist'))   # no error yet - it is checked only when used in a query

Column<'age'>
Column<'age'>
Column<'age'>
Column<'this_column_does_not_exist'>


### 6.2 Selecting and computing columns
`select()` chooses which columns to return (a **projection**). You can compute new values inside it and rename them with `alias()`.

In [21]:
df.select('name').show()
df.select(df['name'], df['age'] + 1).show()
df.select(df['name'], (df['age'] + 1).alias('NewAge')).show()
df.select('name', 'age', F.expr('age * 0.1 AS AgeTenth')).show()

+-------+
|   name|
+-------+
|Michael|
|   Andy|
| Justin|
+-------+

+-------+---------+
|   name|(age + 1)|
+-------+---------+
|Michael|     NULL|
|   Andy|       31|
| Justin|       20|
+-------+---------+

+-------+------+
|   name|NewAge|
+-------+------+
|Michael|  NULL|
|   Andy|    31|
| Justin|    20|
+-------+------+

+-------+----+--------+
|   name| age|AgeTenth|
+-------+----+--------+
|Michael|NULL|    NULL|
|   Andy|  30|     3.0|
| Justin|  19|     1.9|
+-------+----+--------+



In [22]:
blogs_df.select('Hits').show(3)
blogs_df.select((F.col('Hits') * 2).alias('Double Hits')).show()
blogs_df.select('Hits', F.expr('Hits * 2 AS DoubleHits')).show()

+----+
|Hits|
+----+
|4535|
|8908|
|7659|
+----+
only showing top 3 rows
+-----------+
|Double Hits|
+-----------+
|       9070|
|      17816|
|      15318|
|      21136|
|      81156|
|      51136|
+-----------+

+-----+----------+
| Hits|DoubleHits|
+-----+----------+
| 4535|      9070|
| 8908|     17816|
| 7659|     15318|
|10568|     21136|
|40578|     81156|
|25568|     51136|
+-----+----------+



> **Common mistake:** a plain string is treated as a **column name**, not an expression.
> `blogs_df.select('Hits * 2')` looks for a column literally named `"Hits * 2"` and fails. Use `F.expr('Hits * 2')` or `F.col('Hits') * 2` instead.

In [23]:
try:
    blogs_df.select('Hits * 2').show()
except Exception as e:
    print(type(e).__name__, "->", str(e)[:150])

{"ts": "2026-10-30 05:14:39.863", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `Hits * 2` cannot be resolved. Did you mean one of the following? [`Hits`, `First`, `Last`, `Id`, `Url`]. SQLSTATE: 42703", "context": {"file": "java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o234.select.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `Hits * 2` cannot be resolved. Did you mean one of the following? [`Hits`, `First`, `Last`, `Id`, `Url`]. SQLSTATE: 42703;\n'Project ['Hits * 2]\n+- LogicalRDD [Id#62, First#63, Last#64, Url#65, Published#66, Hits#67, Campaigns#68], false\n\n\tat org.apache.spark.sql.errors.Que

AnalysisException -> [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `Hits * 2` cannot be resolved. Did you mean one of the followi


---
## 7. Filtering rows: `filter()` / `where()`
`filter()` and `where()` are **the same method** (two names for the same thing). A condition can be written in two styles:

| Style | Example | Equality | Not equal |
|-------|---------|----------|-----------|
| SQL string | `df.where("age > 21")` | `=` | `!=` or `<>` |
| Column expression | `df.where(F.col('age') > 21)` | `==` | `!=` |

Combine column conditions with `&` (and), `|` (or), `~` (not), and **always wrap each condition in parentheses**.

In [24]:
df.filter(df['age'] > 21).show()
df.where("age > 21").show()
df.where((F.col('age') > 18) & (F.col('name') != 'Andy')).show()
df.where(F.col('age').isNull()).show()       # rows WITH a missing age
df.where(F.col('age').isNotNull()).show()    # rows WITHOUT a missing age

+---+----+
|age|name|
+---+----+
| 30|Andy|
+---+----+

+---+----+
|age|name|
+---+----+
| 30|Andy|
+---+----+

+---+------+
|age|  name|
+---+------+
| 19|Justin|
+---+------+

+----+-------+
| age|   name|
+----+-------+
|NULL|Michael|
+----+-------+

+---+------+
|age|  name|
+---+------+
| 30|  Andy|
| 19|Justin|
+---+------+



---
## 8. Adding, renaming and dropping columns
| Method | Purpose |
|--------|---------|
| `withColumn(name, expression)` | add a new column, or **replace** an existing one with the same name |
| `withColumnRenamed(old, new)` | rename a column |
| `drop(col1, col2, ...)` | remove columns |

Remember: each call returns a **new** DataFrame — the original is unchanged.

In [25]:
# New boolean column
blogs_df.withColumn('Big Hitters', F.col('Hits') > 10000).show()

+---+---------+-------+-----------------+---------+-----+--------------------+-----------+
| Id|    First|   Last|              Url|Published| Hits|           Campaigns|Big Hitters|
+---+---------+-------+-----------------+---------+-----+--------------------+-----------+
|  1|    Jules|  Damji|https://tinyurl.1| 1/4/2016| 4535| [twitter, LinkedIn]|      false|
|  2|   Brooke|  Wenig|https://tinyurl.2| 5/5/2018| 8908| [twitter, LinkedIn]|      false|
|  3|    Denny|    Lee|https://tinyurl.3| 6/7/2019| 7659|[web, twitter, FB...|      false|
|  4|Tathagata|    Das|https://tinyurl.4|5/12/2018|10568|       [twitter, FB]|       true|
|  5|    Matei|Zaharia|https://tinyurl.5|5/14/2014|40578|[web, twitter, FB...|       true|
|  6|  Reynold|    Xin|https://tinyurl.6| 3/2/2015|25568| [twitter, LinkedIn]|       true|
+---+---------+-------+-----------------+---------+-----+--------------------+-----------+



In [26]:
# concat() glues strings together with nothing in between
blogs_df.withColumn("Full Name", F.concat(F.col('First'), F.col('Last'))).select('Full Name').show()

# concat_ws(separator, ...) puts a separator between the values ("ws" = with separator)
blogs_full = blogs_df.withColumn("Full Name", F.concat_ws(' ', F.col('First'), F.col('Last')))
blogs_full.select('Full Name', 'Hits').show()

+------------+
|   Full Name|
+------------+
|  JulesDamji|
| BrookeWenig|
|    DennyLee|
|TathagataDas|
|MateiZaharia|
|  ReynoldXin|
+------------+

+-------------+-----+
|    Full Name| Hits|
+-------------+-----+
|  Jules Damji| 4535|
| Brooke Wenig| 8908|
|    Denny Lee| 7659|
|Tathagata Das|10568|
|Matei Zaharia|40578|
|  Reynold Xin|25568|
+-------------+-----+



In [27]:
# Rename and drop
renamed = blogs_full.withColumnRenamed('Hits', 'TotalHits').drop('Url', 'Campaigns')
renamed.printSchema()
blogs_full.printSchema()      # the original still has Hits, Url, Campaigns (immutability!)

root
 |-- Id: integer (nullable = false)
 |-- First: string (nullable = true)
 |-- Last: string (nullable = true)
 |-- Published: string (nullable = true)
 |-- TotalHits: integer (nullable = true)
 |-- Full Name: string (nullable = false)

root
 |-- Id: integer (nullable = false)
 |-- First: string (nullable = true)
 |-- Last: string (nullable = true)
 |-- Url: string (nullable = true)
 |-- Published: string (nullable = true)
 |-- Hits: integer (nullable = true)
 |-- Campaigns: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- Full Name: string (nullable = false)



---
## 9. Sorting: `sort()` / `orderBy()`
`sort()` and `orderBy()` are the same. Default order is **ascending**. Several ways to sort descending:

In [28]:
blogs_df.sort('Id').show(3)
blogs_df.sort('Id', 'First').show(3)               # by Id, then by First

blogs_df.sort(blogs_df['Id'].desc()).show(3)       # Column.desc()
blogs_df.sort(F.col('Id').desc()).show(3)
blogs_df.sort(F.desc('Id')).show(3)                # F.desc()
blogs_df.orderBy('Id', ascending=False).show(3)    # ascending flag
blogs_df.sort(['Id', 'First'], ascending=[False, True]).show(3)   # one flag per column

+---+------+-----+-----------------+---------+----+--------------------+
| Id| First| Last|              Url|Published|Hits|           Campaigns|
+---+------+-----+-----------------+---------+----+--------------------+
|  1| Jules|Damji|https://tinyurl.1| 1/4/2016|4535| [twitter, LinkedIn]|
|  2|Brooke|Wenig|https://tinyurl.2| 5/5/2018|8908| [twitter, LinkedIn]|
|  3| Denny|  Lee|https://tinyurl.3| 6/7/2019|7659|[web, twitter, FB...|
+---+------+-----+-----------------+---------+----+--------------------+
only showing top 3 rows
+---+------+-----+-----------------+---------+----+--------------------+
| Id| First| Last|              Url|Published|Hits|           Campaigns|
+---+------+-----+-----------------+---------+----+--------------------+
|  1| Jules|Damji|https://tinyurl.1| 1/4/2016|4535| [twitter, LinkedIn]|
|  2|Brooke|Wenig|https://tinyurl.2| 5/5/2018|8908| [twitter, LinkedIn]|
|  3| Denny|  Lee|https://tinyurl.3| 6/7/2019|7659|[web, twitter, FB...|
+---+------+-----+---------

---
## 10. Grouping and aggregation
`groupBy()` follows the **split → apply → combine** strategy:
1. **Split** the rows into groups by a key,
2. **Apply** an aggregate function to each group,
3. **Combine** the results back into a DataFrame.

`groupBy()` alone returns a `GroupedData` object — you must follow it with an aggregation (`count`, `avg`, `sum`, `min`, `max`, `agg`).

In [29]:
fruits = spark.createDataFrame([
    ['red', 'banana', 1, 10], ['blue', 'banana', 2, 20], ['red', 'carrot', 3, 30],
    ['blue', 'grape', 4, 40], ['red', 'carrot', 5, 50], ['black', 'carrot', 6, 60],
    ['red', 'banana', 7, 70], ['red', 'grape', 8, 80]], schema=['color', 'fruit', 'v1', 'v2'])
fruits.show()
print(fruits.groupBy('color'))     # GroupedData - not a DataFrame yet

+-----+------+---+---+
|color| fruit| v1| v2|
+-----+------+---+---+
|  red|banana|  1| 10|
| blue|banana|  2| 20|
|  red|carrot|  3| 30|
| blue| grape|  4| 40|
|  red|carrot|  5| 50|
|black|carrot|  6| 60|
|  red|banana|  7| 70|
|  red| grape|  8| 80|
+-----+------+---+---+

GroupedData[grouping expressions: [color], value: [color: string, fruit: string ... 2 more fields], type: GroupBy]


In [30]:
fruits.groupBy('color').count().show()
fruits.groupBy('color').avg().show()             # average of ALL numeric columns
fruits.groupBy('color').avg('v1').show()         # average of one column
fruits.groupBy('color').avg('v1', 'v2').show()

+-----+-----+
|color|count|
+-----+-----+
|  red|    5|
| blue|    2|
|black|    1|
+-----+-----+

+-----+-------+-------+
|color|avg(v1)|avg(v2)|
+-----+-------+-------+
|  red|    4.8|   48.0|
|black|    6.0|   60.0|
| blue|    3.0|   30.0|
+-----+-------+-------+

+-----+-------+
|color|avg(v1)|
+-----+-------+
|  red|    4.8|
| blue|    3.0|
|black|    6.0|
+-----+-------+

+-----+-------+-------+
|color|avg(v1)|avg(v2)|
+-----+-------+-------+
|  red|    4.8|   48.0|
|black|    6.0|   60.0|
| blue|    3.0|   30.0|
+-----+-------+-------+



### 10.1 `agg()` — several aggregations with your own column names
The shortcut methods (`avg()`, `count()`) produce names like `avg(v1)`. With `agg()` you can mix different functions and rename them with `alias()`.

In [31]:
(fruits.groupBy('color')
       .agg(F.avg('v1').alias('Average v1'),
            F.sum('v2').alias('Sum v2'),
            F.max('v2').alias('Max v2'),
            F.count('*').alias('Rows'))
       .sort('Average v1')
       .show())

+-----+----------+------+------+----+
|color|Average v1|Sum v2|Max v2|Rows|
+-----+----------+------+------+----+
| blue|       3.0|    60|    40|   2|
|  red|       4.8|   240|    80|   5|
|black|       6.0|    60|    60|   1|
+-----+----------+------+------+----+



In [32]:
# Group by more than one column
fruits.groupBy('color', 'fruit').agg(F.avg('v1').alias('v1_avg')).sort('color', 'fruit').show()

+-----+------+------+
|color| fruit|v1_avg|
+-----+------+------+
|black|carrot|   6.0|
| blue|banana|   2.0|
| blue| grape|   4.0|
|  red|banana|   4.0|
|  red|carrot|   4.0|
|  red| grape|   8.0|
+-----+------+------+



### 10.2 Aggregating the whole DataFrame (no grouping)

In [33]:
fruits.agg(F.min('v1').alias('Min v1'), F.max('v1').alias('Max v1'), F.avg('v2').alias('Avg v2')).show()

# Same thing written with select()
fruits.select(F.min('v1'), F.max('v1'), F.avg('v2')).show()

+------+------+------+
|Min v1|Max v1|Avg v2|
+------+------+------+
|     1|     8|  45.0|
+------+------+------+

+-------+-------+-------+
|min(v1)|max(v1)|avg(v2)|
+-------+-------+-------+
|      1|      8|   45.0|
+-------+-------+-------+



> **Note:** `fruits.toPandas()` converts a Spark DataFrame to a **pandas** DataFrame on the driver — handy for small results, but like `collect()` it loads everything into the driver's memory.

---
## 11. Conditional columns: `when()` / `otherwise()`
This is the DataFrame version of SQL's `CASE WHEN ... THEN ... ELSE ... END`.
Conditions are checked **in order** and the first one that is true wins; `otherwise()` is the `ELSE`.

We use the US flight delays dataset (`delay` = minutes late; negative = left early).
Reading a CSV with a schema and `header=True` is explained in detail in section 13.

In [68]:
flights_schema = "date STRING, delay INT, distance INT, origin STRING, destination STRING"
flights = spark.read.csv(f"{DATA_DIR}/departuredelays.csv", schema=flights_schema, header=True)
flights.show(3)

+--------+-----+--------+------+-----------+
|    date|delay|distance|origin|destination|
+--------+-----+--------+------+-----------+
|01011245|    6|     602|   ABE|        ATL|
|01020600|   -8|     369|   ABE|        DTW|
|01021245|   -2|     602|   ABE|        ATL|
+--------+-----+--------+------+-----------+
only showing top 3 rows


In [69]:
flights_labeled = flights.select(
    'delay', 'origin', 'destination',
    F.when(F.col('delay') > 360, 'Very Long Delays')
     .when(F.col('delay') > 120, 'Long Delays')
     .when(F.col('delay') > 60,  'Short Delays')
     .when(F.col('delay') > 0,   'Tolerable Delays')
     .when(F.col('delay') == 0,  'No Delays')
     .otherwise('Early')
     .alias('Flight_Delays'))

flights_labeled.show(10)
flights_labeled.groupBy('Flight_Delays').count().orderBy(F.desc('count')).show()

+-----+------+-----------+----------------+
|delay|origin|destination|   Flight_Delays|
+-----+------+-----------+----------------+
|    6|   ABE|        ATL|Tolerable Delays|
|   -8|   ABE|        DTW|           Early|
|   -2|   ABE|        ATL|           Early|
|   -4|   ABE|        ATL|           Early|
|   -4|   ABE|        ATL|           Early|
|    0|   ABE|        ATL|       No Delays|
|   10|   ABE|        ATL|Tolerable Delays|
|   28|   ABE|        ATL|Tolerable Delays|
|   88|   ABE|        ATL|    Short Delays|
|    9|   ABE|        ATL|Tolerable Delays|
+-----+------+-----------+----------------+
only showing top 10 rows


+----------------+------+
|   Flight_Delays| count|
+----------------+------+
|           Early|668729|
|Tolerable Delays|497742|
|       No Delays|131122|
|    Short Delays| 61039|
|     Long Delays| 31447|
|Very Long Delays|  1499|
+----------------+------+



> Because the conditions are checked in order, `.when(delay > 120)` only sees rows that were **not** already `> 360`, so we don't need `delay > 120 AND delay < 360`.
> You may also see the older, nested style `F.when(...).otherwise(F.when(...).otherwise(...))` — it gives the same result but is harder to read.

---
## 12. User-Defined Functions (UDFs)
Spark has hundreds of built-in functions, but sometimes you need your own logic. A **UDF** wraps a normal Python function so Spark can apply it to **every row** of a column.

Bonus: once registered, a UDF can also be used inside **Spark SQL** (see the SQL notebook) — e.g. a data scientist can wrap an ML model in a UDF and an analyst can call it from SQL.

In [70]:
names_df = spark.createDataFrame([("1", "john jones"), ("2", "tracey smith"), ("3", "amy sanders")],
                                 schema=["Seqno", "Name"])
names_df.show(truncate=False)

+-----+------------+
|Seqno|Name        |
+-----+------------+
|1    |john jones  |
|2    |tracey smith|
|3    |amy sanders |
+-----+------------+



In [71]:
def convert_case(text):
    """Capitalize the first letter of every word: 'john jones' -> 'John Jones'."""
    if text is None:                  # UDFs receive None for null values - always handle it
        return None
    return " ".join(word[:1].upper() + word[1:] for word in text.split(" "))

convert_case('ahmed mohamed hamed')   # works on a normal Python string

'Ahmed Mohamed Hamed'

Calling the plain Python function on a **column** does **not** work: `names_df['Name']` is not a string, it is a *Column object* (a description of a column, not its values), so `text.split(" ")` inside the function fails.

In [72]:
try:
    names_df.select(convert_case(names_df['Name']))
except Exception as e:
    print(type(e).__name__, "->", e)

TypeError -> 'Column' object is not callable


**Option 1 — wrap it with `F.udf(function, returnType)`:**

In [73]:
convert_udf = F.udf(convert_case, StringType())
names_df.select('Seqno', convert_udf(F.col('Name')).alias('Name')).show(truncate=False)

+-----+------------+
|Seqno|Name        |
+-----+------------+
|1    |John Jones  |
|2    |Tracey Smith|
|3    |Amy Sanders |
+-----+------------+



**Option 2 — the `@udf` decorator:**

In [74]:
@F.udf(returnType=StringType())
def convert_case_annot(text):
    if text is None:
        return None
    return " ".join(word[:1].upper() + word[1:] for word in text.split(" "))

names_df.select('Seqno', convert_case_annot(F.col('Name')).alias('Name')).show(truncate=False)

+-----+------------+
|Seqno|Name        |
+-----+------------+
|1    |John Jones  |
|2    |Tracey Smith|
|3    |Amy Sanders |
+-----+------------+



> **Performance:** Python UDFs are **slow**. Spark runs on the JVM, so every row must be sent to a Python process, converted, processed and sent back.
> **Always check for a built-in function first** — here `F.initcap()` does exactly the same job and runs inside the JVM:

In [75]:
names_df.select('Seqno', F.initcap('Name').alias('Name')).show(truncate=False)

+-----+------------+
|Seqno|Name        |
+-----+------------+
|1    |John Jones  |
|2    |Tracey Smith|
|3    |Amy Sanders |
+-----+------------+



---
## 13. Reading CSV files correctly

### 13.1 `header`, `inferSchema` and `schema`
| Option | Default | Meaning |
|--------|---------|---------|
| `header=True` | `False` | first line holds column names. Without it, columns are named `_c0, _c1, ...` and the header line becomes a **data row** |
| `inferSchema=True` | `False` | scan the data to guess types. Without it, **every column is a string** |
| `schema=...` | — | give the types yourself (fastest, safest) |
| `samplingRatio=0.01` | `1.0` | infer the schema from only a **sample** of the rows (cheaper on big files) |

In [39]:
null_path = f"{DATA_DIR}/NullData.csv"

spark.read.csv(null_path).printSchema()                                   # _c0.. and all strings
spark.read.csv(null_path, header=True).printSchema()                      # names OK, all strings
spark.read.csv(null_path, header=True, inferSchema=True).printSchema()    # names + real types

root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)

root
 |-- Id: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sales: string (nullable = true)



root
 |-- Id: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sales: double (nullable = true)



In [40]:
# Our own schema. Because we give the column names in the schema, we still need header=True
# so Spark SKIPS the header line instead of reading it as data.
s = 'Id STRING, Name STRING, Sales DOUBLE'
spark.read.csv(null_path, schema=s, header=True).show()

+----+-----+-----+
|  Id| Name|Sales|
+----+-----+-----+
|emp1| John| NULL|
|emp2| NULL| NULL|
|emp3| NULL|345.0|
|emp4|Cindy|456.0|
+----+-----+-----+



### 13.2 Read modes — what to do with malformed rows
| Mode | Behaviour on a bad row |
|------|------------------------|
| `PERMISSIVE` (default) | keep the row, put `null` in the fields that cannot be parsed |
| `DROPMALFORMED` | silently drop the bad rows |
| `FAILFAST` | stop immediately with an error |

`people_bad.csv` has one row where the age is not a number:

In [41]:
bad_path = f"{DATA_DIR}/people_bad.csv"
spark.read.text(bad_path).show()          # raw lines: 'Ayman,twelve' is the bad row

s = 'Name STRING, Age INT'
spark.read.csv(bad_path, schema=s, mode='PERMISSIVE').show()
spark.read.csv(bad_path, schema=s, mode='DROPMALFORMED').show()

try:
    spark.read.csv(bad_path, schema=s, mode='FAILFAST').show()
except Exception as e:
    print("FAILFAST stopped the read:", type(e).__name__)

+------------+
|       value|
+------------+
|    Ahmed,20|
|  Mohamed,25|
|Ayman,twelve|
|     Sara,30|
+------------+

+-------+----+
|   Name| Age|
+-------+----+
|  Ahmed|  20|
|Mohamed|  25|
|  Ayman|NULL|
|   Sara|  30|
+-------+----+

+-------+---+
|   Name|Age|
+-------+---+
|  Ahmed| 20|
|Mohamed| 25|
|   Sara| 30|
+-------+---+

FAILFAST stopped the read: Py4JJavaError


26/10/30 05:14:46 ERROR TaskSetManager: Task 0 in stage 110.0 failed 4 times; aborting job


---
## 14. Missing data
Three strategies for `null` values:
1. **Keep** them,
2. **Remove** them → `df.na.drop(...)`,
3. **Fill** them with some value → `df.na.fill(...)`.

In [42]:
df = spark.read.csv(f"{DATA_DIR}/NullData.csv", header=True, inferSchema=True)
df.show()
df.printSchema()

+----+-----+-----+
|  Id| Name|Sales|
+----+-----+-----+
|emp1| John| NULL|
|emp2| NULL| NULL|
|emp3| NULL|345.0|
|emp4|Cindy|456.0|
+----+-----+-----+

root
 |-- Id: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sales: double (nullable = true)



### 14.1 `na.drop(how, thresh, subset)`
| Parameter | Meaning |
|-----------|---------|
| `how='any'` (default) | drop a row if it has **any** null |
| `how='all'` | drop a row only if **all** its values are null |
| `thresh=n` | keep only rows that have **at least `n` non-null** values (overrides `how`) |
| `subset=[...]` | only look at these columns |

In [43]:
df.na.drop().show()                          # same as how='any': only emp4 survives
df.na.drop(how='all').show()                 # no row is completely empty -> nothing dropped
df.na.drop(thresh=2).show()                  # need at least 2 non-null values

+----+-----+-----+
|  Id| Name|Sales|
+----+-----+-----+
|emp4|Cindy|456.0|
+----+-----+-----+

+----+-----+-----+
|  Id| Name|Sales|
+----+-----+-----+
|emp1| John| NULL|
|emp2| NULL| NULL|
|emp3| NULL|345.0|
|emp4|Cindy|456.0|
+----+-----+-----+

+----+-----+-----+
|  Id| Name|Sales|
+----+-----+-----+
|emp1| John| NULL|
|emp3| NULL|345.0|
|emp4|Cindy|456.0|
+----+-----+-----+



In [44]:
df.na.drop(subset=['Sales']).show()                     # only rows where Sales is missing are dropped
df.na.drop(subset=['Name', 'Sales'], how='all').show()  # drop if BOTH Name and Sales are missing
df.na.drop(subset=['Name', 'Sales'], thresh=1).show()   # same idea, using thresh

+----+-----+-----+
|  Id| Name|Sales|
+----+-----+-----+
|emp3| NULL|345.0|
|emp4|Cindy|456.0|
+----+-----+-----+

+----+-----+-----+
|  Id| Name|Sales|
+----+-----+-----+
|emp1| John| NULL|
|emp3| NULL|345.0|
|emp4|Cindy|456.0|
+----+-----+-----+

+----+-----+-----+
|  Id| Name|Sales|
+----+-----+-----+
|emp1| John| NULL|
|emp3| NULL|345.0|
|emp4|Cindy|456.0|
+----+-----+-----+



### 14.2 `na.fill(value, subset)`
Spark only fills columns whose **type matches** the value: a string fills string columns, a number fills numeric columns. Other columns are silently ignored.

In [45]:
df.na.fill('No Name').show()                    # fills only string columns (Name)
df.na.fill(0).show()                            # fills only numeric columns (Sales)
df.na.fill('No Name', subset=['Name']).show()
df.na.fill(0, subset=['Name']).show()           # ignored: Name is a string, 0 is a number

+----+-------+-----+
|  Id|   Name|Sales|
+----+-------+-----+
|emp1|   John| NULL|
|emp2|No Name| NULL|
|emp3|No Name|345.0|
|emp4|  Cindy|456.0|
+----+-------+-----+

+----+-----+-----+
|  Id| Name|Sales|
+----+-----+-----+
|emp1| John|  0.0|
|emp2| NULL|  0.0|
|emp3| NULL|345.0|
|emp4|Cindy|456.0|
+----+-----+-----+

+----+-------+-----+
|  Id|   Name|Sales|
+----+-------+-----+
|emp1|   John| NULL|
|emp2|No Name| NULL|
|emp3|No Name|345.0|
|emp4|  Cindy|456.0|
+----+-------+-----+

+----+-----+-----+
|  Id| Name|Sales|
+----+-----+-----+
|emp1| John| NULL|
|emp2| NULL| NULL|
|emp3| NULL|345.0|
|emp4|Cindy|456.0|
+----+-----+-----+



### 14.3 Filling with the mean (a common technique)
1. compute the mean with an aggregation,
2. bring that single number back to Python with `collect()`,
3. use it in `na.fill`.

In [46]:
mean_row = df.select(F.mean('Sales').alias('mean')).collect()   # [Row(mean=400.5)]
mean_sales = mean_row[0].mean
print("mean sales =", mean_sales)

df.na.fill(mean_sales, subset=['Sales']).show()

mean sales = 400.5
+----+-----+-----+
|  Id| Name|Sales|
+----+-----+-----+
|emp1| John|400.5|
|emp2| NULL|400.5|
|emp3| NULL|345.0|
|emp4|Cindy|456.0|
+----+-----+-----+



In [47]:
# Fill different columns with different values in one call, using a dict
df.na.fill({'Name': 'NoName', 'Sales': mean_sales}).show()

+----+------+-----+
|  Id|  Name|Sales|
+----+------+-----+
|emp1|  John|400.5|
|emp2|NoName|400.5|
|emp3|NoName|345.0|
|emp4| Cindy|456.0|
+----+------+-----+



---
## 15. Data sources: `DataFrameReader` and `DataFrameWriter`

### 15.1 DataFrameReader
Recommended pattern:
```
spark.read.format(...).option("key", "value").schema(...).load(path)
```
- You get a reader only through the SparkSession: `spark.read` (static data) or `spark.readStream` (streaming data).
- Shortcuts exist for each format: `spark.read.csv(...)`, `spark.read.json(...)`, `spark.read.parquet(...)`.

| Method | Arguments | Notes |
|--------|-----------|-------|
| `format()` | `"parquet"`, `"csv"`, `"json"`, `"jdbc"`, `"orc"`, `"avro"`, `"text"` | default is **parquet** |
| `option()` | `("mode", "PERMISSIVE/FAILFAST/DROPMALFORMED")`, `("inferSchema", "true")`, `("header", "true")`, `("path", "...")` | key/value pairs |
| `schema()` | DDL string or `StructType` | |
| `load()` | path | can be a file, a folder, or a wildcard like `csv/*` |

In [76]:
# Parquet: the schema is stored INSIDE the file, so no inferSchema / header needed
df_parquet = spark.read.format('parquet').load(f"{DATA_DIR}/parquet/2010-summary.parquet")
df_parquet = spark.read.parquet(f"{DATA_DIR}/parquet/2010-summary.parquet")      # shortcut
df_parquet.printSchema()
df_parquet.show(5)

root
 |-- DEST_COUNTRY_NAME: string (nullable = true)
 |-- ORIGIN_COUNTRY_NAME: string (nullable = true)
 |-- count: long (nullable = true)

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Romania|    1|
|    United States|            Ireland|  264|
|    United States|              India|   69|
|            Egypt|      United States|   24|
|Equatorial Guinea|      United States|    1|
+-----------------+-------------------+-----+
only showing top 5 rows


In [77]:
# CSV: a wildcard reads every file in the folder into ONE DataFrame
df_csv = (spark.read.format('csv')
          .option('header', 'true')
          .option('inferSchema', 'true')
          .load(f"{DATA_DIR}/csv/*"))
print("rows from all csv files:", df_csv.count())

# Same thing with the shortcut
df_csv = spark.read.csv(f"{DATA_DIR}/csv/*", header=True, inferSchema=True)

rows from all csv files: 1502


In [78]:
# CSV with an explicit schema and extra options
summary_schema = "DEST_COUNTRY_NAME STRING, ORIGIN_COUNTRY_NAME STRING, count INT"
df_csv = (spark.read.format("csv")
          .option("header", "true")
          .schema(summary_schema)
          .option("mode", "FAILFAST")      # stop on any bad row
          .option("nullValue", "")         # treat empty fields as null
          .load(f"{DATA_DIR}/csv/*"))
df_csv.show(3)

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Romania|    1|
|    United States|            Ireland|  264|
|    United States|              India|   69|
+-----------------+-------------------+-----+
only showing top 3 rows


In [79]:
# JSON: schema is inferred automatically (header/inferSchema options don't apply)
df_json = spark.read.format('json').load(f"{DATA_DIR}/json/*")
df_json.show(3)

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Romania|   15|
|    United States|            Croatia|    1|
|    United States|            Ireland|  344|
+-----------------+-------------------+-----+
only showing top 3 rows


> **JSON single-line vs multi-line:** by default each line must be one JSON object. If one JSON object spans several lines, add `.option("multiLine", "true")`.

### 15.2 DataFrameWriter
You get a writer **from the DataFrame** you want to save: `df.write` (static) or `df.writeStream` (streaming).
```
df.write.format(...).option(...).partitionBy(...).bucketBy(...).mode(...).save(path)
df.write.format(...).option(...).sortBy(...).saveAsTable(table_name)
```
**Save modes** — what happens if the destination already exists:

| Mode | Behaviour |
|------|-----------|
| `error` / `errorifexists` (default) | raise an error |
| `overwrite` | replace the existing data |
| `append` | add to the existing data |
| `ignore` | do nothing, silently |

> Spark writes a **folder**, not a single file: it contains one `part-...` file per partition plus a `_SUCCESS` marker.

In [80]:
flights.write.format('json').mode('overwrite').save(f"{OUT_DIR}/flights_json")
flights.write.json(f"{OUT_DIR}/flights_json", mode='overwrite')          # shortcut, same result

flights.write.format('csv').mode('overwrite').option('header', 'true').save(f"{OUT_DIR}/flights_csv")

(flights.write.format('parquet')
        .mode('overwrite')
        .option('compression', 'snappy')
        .save(f"{OUT_DIR}/flights_parquet"))

# Read back what we wrote
spark.read.parquet(f"{OUT_DIR}/flights_parquet").show(3)

+--------+-----+--------+------+-----------+
|    date|delay|distance|origin|destination|
+--------+-----+--------+------+-----------+
|01011245|    6|     602|   ABE|        ATL|
|01020600|   -8|     369|   ABE|        DTW|
|01021245|   -2|     602|   ABE|        ATL|
+--------+-----+--------+------+-----------+
only showing top 3 rows


See the folders Spark created from a terminal with `hdfs dfs -ls /user/user/data/spark_course_output/flights_parquet` — one `part-...` file per partition plus `_SUCCESS`.

> `df.write.saveAsTable('name')` saves the DataFrame as a **table** in Spark's catalog — this is covered in detail in the **SQL notebook** (managed vs unmanaged tables).

### 15.3 Parquet — the recommended format
- **Default** data source in Spark, and supported by most big-data tools.
- **Columnar**: values of each column are stored together, so a query that needs 2 of 30 columns reads only those 2. Aggregations become much faster than on row-based files like CSV.
- **Compressed** (e.g. snappy) — often at least one-third smaller than CSV, which saves storage costs (S3 / GCS charge per GB stored; Athena / Spectrum charge per GB scanned).
- **Self-describing**: the schema is stored in the file. You can still pass a schema, to make sure of the types and to read slightly faster.
- A Parquet "file" is really a **directory** with data files, metadata and status files.

*Recommendation: after you clean and transform your data, save it as Parquet for the next steps of your pipeline.*

In [81]:
# Parquet with an explicit schema
parq_schema = 'DEST_COUNTRY_NAME STRING, ORIGIN_COUNTRY_NAME STRING, count LONG'
spark.read.format('parquet').schema(parq_schema).load(f"{DATA_DIR}/parquet/2010-summary.parquet").printSchema()

root
 |-- DEST_COUNTRY_NAME: string (nullable = true)
 |-- ORIGIN_COUNTRY_NAME: string (nullable = true)
 |-- count: long (nullable = true)



### 15.4 Databases over JDBC (PostgreSQL, MySQL, SQL Server)
Spark can read a database table into a DataFrame (and write a DataFrame to a table) through **JDBC**.
You get all the DataFrame benefits on database data, including joining it with files.

**Requirement:** the database's JDBC driver `.jar` must be available to Spark, either copied into Spark's `jars/` folder (e.g. `/opt/spark/jars`) or passed when the session is **first** created (restart the kernel if a session already exists):
```python
spark = SparkSession.builder.config("spark.jars", "postgresql-42.6.0.jar").getOrCreate()
```

> **Never write a real password directly in a notebook** you share with others. Read it from an environment variable or a config file that is not shared. The cell below uses placeholders.

In [82]:
import os

jdbc_url  = "jdbc:postgresql://localhost:5432/<database>"     # jdbc:postgresql://<host>:<port>/<database>
jdbc_user = os.environ.get("PG_USER", "<user>")
jdbc_pass = os.environ.get("PG_PASSWORD", "<password>")

RUN_JDBC = False     # set to True when you have a PostgreSQL server and the driver jar

if RUN_JDBC:
    # Write a DataFrame into a database table
    (flights.write.format("jdbc")
            .mode("overwrite")
            .option("url", jdbc_url)
            .option("dbtable", "departuredelays")
            .option("user", jdbc_user)
            .option("password", jdbc_pass)
            .option("driver", "org.postgresql.Driver")
            .save())

    # Read a database table into a DataFrame
    df_from_db = (spark.read.format("jdbc")
                  .option("url", jdbc_url)
                  .option("dbtable", "departuredelays")
                  .option("user", jdbc_user)
                  .option("password", jdbc_pass)
                  .option("driver", "org.postgresql.Driver")
                  .load())
    df_from_db.show(5)

There is also a shorter `jdbc()` method:
```python
df = spark.read.jdbc(url, "schema.table", properties={"user": user, "password": pwd})
df.write.jdbc(url, "schema.table", mode="overwrite", properties={"user": user, "password": pwd})
```

Connection URLs and drivers for other databases:

| Database | URL format | Driver class |
|----------|-----------|--------------|
| PostgreSQL | `jdbc:postgresql://host:5432/db` | `org.postgresql.Driver` |
| MySQL | `jdbc:mysql://host:3306/db` | `com.mysql.cj.jdbc.Driver` |
| MS SQL Server | `jdbc:sqlserver://host:1433;database=db` | `com.microsoft.sqlserver.jdbc.SQLServerDriver` |

Spark can also connect (through extra connectors) to Azure Cosmos DB, Apache Cassandra, Snowflake, MongoDB and many others.

---
## 16. Caching
Because of lazy evaluation, **every action recomputes the DataFrame from the source** (re-reads the file, re-applies every transformation).
If you will use the same DataFrame several times, keep it in memory:

| Call | Effect |
|------|--------|
| `df.cache()` | mark the DataFrame to be kept in memory. It is **lazy**: the data is stored during the **next action**. |
| `df.unpersist()` | remove this DataFrame from the cache |
| `spark.catalog.clearCache()` | remove **everything** from the cache |

Use `%%time` (a Jupyter magic, must be the first line of the cell) to measure the difference.

In [34]:
%%time
fruits.groupBy('color').avg().show()      # before caching

+-----+-------+-------+
|color|avg(v1)|avg(v2)|
+-----+-------+-------+
|  red|    4.8|   48.0|
| blue|    3.0|   30.0|
|black|    6.0|   60.0|
+-----+-------+-------+

CPU times: user 2.63 ms, sys: 773 µs, total: 3.4 ms
Wall time: 180 ms


In [35]:
fruits.cache()

DataFrame[color: string, fruit: string, v1: bigint, v2: bigint]

In [36]:
%%time
fruits.groupBy('color').avg().show()      # 1st action after cache(): computes AND fills the cache

+-----+-------+-------+
|color|avg(v1)|avg(v2)|
+-----+-------+-------+
|  red|    4.8|   48.0|
|black|    6.0|   60.0|
| blue|    3.0|   30.0|
+-----+-------+-------+

CPU times: user 63 µs, sys: 2.95 ms, total: 3.01 ms
Wall time: 446 ms


In [37]:
%%time
fruits.groupBy('color').avg().show()      # 2nd action: served from memory -> faster

+-----+-------+-------+
|color|avg(v1)|avg(v2)|
+-----+-------+-------+
|  red|    4.8|   48.0|
|black|    6.0|   60.0|
| blue|    3.0|   30.0|
+-----+-------+-------+

CPU times: user 4 ms, sys: 4 ms, total: 8.01 ms
Wall time: 142 ms


In [38]:
fruits.unpersist()                 # drop this DataFrame from the cache
spark.catalog.clearCache()         # or drop everything

> On a tiny DataFrame the time difference is small; the benefit is huge on large files (see the fire-calls case study in the next section).

---
## 17. Case study: San Francisco Fire Department calls
A real dataset (~175,000 rows, 28 columns). We will practise everything above: schema inference, projections, filters, renaming, type conversion and aggregations.

### 17.1 Load the data
With a big file, `inferSchema=True` reads the whole file once just to guess the types. `samplingRatio` makes Spark guess from a sample instead:

In [48]:
%%time
fire_df = spark.read.csv(f"{DATA_DIR}/sf-fire-calls.csv", header=True, inferSchema=True)

CPU times: user 0 ns, sys: 2.16 ms, total: 2.16 ms
Wall time: 772 ms


In [49]:
%%time
fire_df = spark.read.csv(f"{DATA_DIR}/sf-fire-calls.csv", header=True, inferSchema=True, samplingRatio=0.01)

CPU times: user 1.03 ms, sys: 328 µs, total: 1.36 ms
Wall time: 225 ms


In [50]:
fire_df.cache()
print("rows:", fire_df.count())    # this first action also fills the cache
fire_df.printSchema()

rows: 175296
root
 |-- CallNumber: integer (nullable = true)
 |-- UnitID: string (nullable = true)
 |-- IncidentNumber: integer (nullable = true)
 |-- CallType: string (nullable = true)
 |-- CallDate: string (nullable = true)
 |-- WatchDate: string (nullable = true)
 |-- CallFinalDisposition: string (nullable = true)
 |-- AvailableDtTm: string (nullable = true)
 |-- Address: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Zipcode: integer (nullable = true)
 |-- Battalion: string (nullable = true)
 |-- StationArea: string (nullable = true)
 |-- Box: integer (nullable = true)
 |-- OriginalPriority: string (nullable = true)
 |-- Priority: string (nullable = true)
 |-- FinalPriority: integer (nullable = true)
 |-- ALSUnit: boolean (nullable = true)
 |-- CallTypeGroup: string (nullable = true)
 |-- NumAlarms: integer (nullable = true)
 |-- UnitType: string (nullable = true)
 |-- UnitSequenceInCallDispatch: integer (nullable = true)
 |-- FirePreventionDistrict: string (null

Things to check when you meet a new dataset: *Are the columns the right types? Do some need converting? Do they contain nulls?*
Notice that `CallDate`, `WatchDate` and `AvailableDtTm` are **strings**, not dates — we will fix that in 17.4.

In [51]:
fire_df.show(5, truncate=False)
fire_df.summary().select('summary', 'Delay', 'NumAlarms').show()

+----------+------+--------------+----------------+----------+----------+--------------------+----------------------+---------------------------+----+-------+---------+-----------+----+----------------+--------+-------------+-------+-------------+---------+--------+--------------------------+----------------------+------------------+---------------------+-------------------------------------+-------------+---------+
|CallNumber|UnitID|IncidentNumber|CallType        |CallDate  |WatchDate |CallFinalDisposition|AvailableDtTm         |Address                    |City|Zipcode|Battalion|StationArea|Box |OriginalPriority|Priority|FinalPriority|ALSUnit|CallTypeGroup|NumAlarms|UnitType|UnitSequenceInCallDispatch|FirePreventionDistrict|SupervisorDistrict|Neighborhood         |Location                             |RowID        |Delay    |
+----------+------+--------------+----------------+----------+----------+--------------------+----------------------+---------------------------+----+-------+--

### 17.2 Projections and filters
- **Projection** = which **columns** to return → `select()`.
- **Filter** = which **rows** to return → `filter()` / `where()`.

In [52]:
fire_df.select('CallNumber').where('Delay > 5').show(5)

# A filter can use a column we created in the same select
fire_df.select('CallNumber', (F.col('Delay') * 2).alias('DblDelay')).where('DblDelay > 5').show(5)

+----------+
|CallNumber|
+----------+
|  20110315|
|  20120147|
|  20130013|
|  20140067|
|  20140177|
+----------+
only showing top 5 rows
+----------+---------+
|CallNumber| DblDelay|
+----------+---------+
|  20110016|      5.9|
|  20110022|      9.4|
|  20110043|6.9666666|
|  20110125|5.4333334|
|  20110215|5.5333334|
+----------+---------+
only showing top 5 rows


In [53]:
# Three equivalent ways to say "CallType is not Medical Incident"
few_fire_df = (fire_df
               .select("IncidentNumber", "AvailableDtTm", "CallType")
               .where(F.col("CallType") != "Medical Incident"))
few_fire_df.show(5, truncate=False)

fire_df.select("IncidentNumber", "AvailableDtTm", "CallType").where("CallType != 'Medical Incident'").show(5, truncate=False)
fire_df.select("IncidentNumber", "AvailableDtTm", "CallType").where(fire_df["CallType"] != "Medical Incident").show(5, truncate=False)

+--------------+----------------------+--------------+
|IncidentNumber|AvailableDtTm         |CallType      |
+--------------+----------------------+--------------+
|2003235       |01/11/2002 01:51:44 AM|Structure Fire|
|2003250       |01/11/2002 04:16:46 AM|Vehicle Fire  |
|2003259       |01/11/2002 06:01:58 AM|Alarms        |
|2003279       |01/11/2002 08:03:26 AM|Structure Fire|
|2003301       |01/11/2002 09:46:44 AM|Alarms        |
+--------------+----------------------+--------------+
only showing top 5 rows
+--------------+----------------------+--------------+
|IncidentNumber|AvailableDtTm         |CallType      |
+--------------+----------------------+--------------+
|2003235       |01/11/2002 01:51:44 AM|Structure Fire|
|2003250       |01/11/2002 04:16:46 AM|Vehicle Fire  |
|2003259       |01/11/2002 06:01:58 AM|Alarms        |
|2003279       |01/11/2002 08:03:26 AM|Structure Fire|
|2003301       |01/11/2002 09:46:44 AM|Alarms        |
+--------------+----------------------+--

In [54]:
# Equality: '=' inside a SQL string, '==' in a column expression
fire_df.select("IncidentNumber", "CallType").where("CallType = 'Medical Incident'").show(3, truncate=False)
fire_df.select("IncidentNumber", "CallType").where(fire_df["CallType"] == "Medical Incident").show(3, truncate=False)

+--------------+----------------+
|IncidentNumber|CallType        |
+--------------+----------------+
|2003241       |Medical Incident|
|2003242       |Medical Incident|
|2003343       |Medical Incident|
+--------------+----------------+
only showing top 3 rows
+--------------+----------------+
|IncidentNumber|CallType        |
+--------------+----------------+
|2003241       |Medical Incident|
|2003242       |Medical Incident|
|2003343       |Medical Incident|
+--------------+----------------+
only showing top 3 rows


### 17.3 Distinct values
*Which call types exist, and how many are there?*

In [55]:
call_types = (fire_df.select('CallType')
                     .where(F.col('CallType').isNotNull())
                     .distinct())
call_types.sort('CallType').show(30, truncate=False)

print("distinct call types:", call_types.count())

+--------------------------------------------+
|CallType                                    |
+--------------------------------------------+
|Administrative                              |
|Aircraft Emergency                          |
|Alarms                                      |
|Assist Police                               |
|Citizen Assist / Service Call               |
|Confined Space / Structure Collapse         |
|Electrical Hazard                           |
|Elevator / Escalator Rescue                 |
|Explosion                                   |
|Extrication / Entrapped (Machinery, Vehicle)|
|Fuel Spill                                  |
|Gas Leak (Natural and LP Gases)             |
|HazMat                                      |
|High Angle Rescue                           |
|Industrial Accidents                        |
|Marine Fire                                 |
|Medical Incident                            |
|Mutual Aid / Assist Outside Agency          |
|Odor (Strang

In [56]:
# Same answer as an aggregation, returned as a DataFrame
fire_df.agg(F.countDistinct('CallType').alias('DistinctCallTypes')).show()

+-----------------+
|DistinctCallTypes|
+-----------------+
|               30|
+-----------------+



### 17.4 Renaming and changing column types
**Rename** `Delay` → `ResponseDelayedMins` so the meaning is clear:

In [57]:
new_fire_df = fire_df.withColumnRenamed('Delay', 'ResponseDelayedMins')
new_fire_df.select('ResponseDelayedMins').where(F.col('ResponseDelayedMins') > 5).show(5)

+-------------------+
|ResponseDelayedMins|
+-------------------+
|               5.35|
|               6.25|
|                5.2|
|                5.6|
|               7.25|
+-------------------+
only showing top 5 rows


**Convert strings to timestamps** with `F.to_timestamp(column, format)`. The format describes how the text looks:

| Pattern | Meaning | Example |
|---------|---------|---------|
| `MM` | month | 01 |
| `dd` | day | 11 |
| `yyyy` | year | 2002 |
| `hh` | hour (1-12) | 01 |
| `mm` | minutes | 51 |
| `ss` | seconds | 44 |
| `a` | AM / PM | AM |

Steps: create the new typed column with `withColumn`, then `drop` the old string column.

In [58]:
fire_ts_df = (new_fire_df
              .withColumn('IncidentDate', F.to_timestamp(F.col('CallDate'), 'MM/dd/yyyy'))
              .drop('CallDate')
              .withColumn('OnWatchDate', F.to_timestamp(F.col('WatchDate'), 'MM/dd/yyyy'))
              .drop('WatchDate')
              .withColumn('AvailableDtTS', F.to_timestamp(F.col('AvailableDtTm'), 'MM/dd/yyyy hh:mm:ss a'))
              .drop('AvailableDtTm'))

fire_ts_df.select('IncidentDate', 'OnWatchDate', 'AvailableDtTS').show(5, truncate=False)

+-------------------+-------------------+-------------------+
|IncidentDate       |OnWatchDate        |AvailableDtTS      |
+-------------------+-------------------+-------------------+
|2002-01-11 00:00:00|2002-01-10 00:00:00|2002-01-11 01:51:44|
|2002-01-11 00:00:00|2002-01-10 00:00:00|2002-01-11 03:01:18|
|2002-01-11 00:00:00|2002-01-10 00:00:00|2002-01-11 02:39:50|
|2002-01-11 00:00:00|2002-01-10 00:00:00|2002-01-11 04:16:46|
|2002-01-11 00:00:00|2002-01-10 00:00:00|2002-01-11 06:01:58|
+-------------------+-------------------+-------------------+
only showing top 5 rows


In [59]:
# Alternative: keep the SAME column names - withColumn with an existing name replaces that column
fire_ts_df_2 = (new_fire_df
                .withColumn('CallDate', F.to_timestamp(F.col('CallDate'), 'MM/dd/yyyy'))
                .withColumn('WatchDate', F.to_timestamp(F.col('WatchDate'), 'MM/dd/yyyy'))
                .withColumn('AvailableDtTm', F.to_timestamp(F.col('AvailableDtTm'), 'MM/dd/yyyy hh:mm:ss a')))
fire_ts_df_2.select('CallDate', 'WatchDate', 'AvailableDtTm').printSchema()

root
 |-- CallDate: timestamp (nullable = true)
 |-- WatchDate: timestamp (nullable = true)
 |-- AvailableDtTm: timestamp (nullable = true)



**`cast()`** converts a column to another type:

In [60]:
fire_cast = new_fire_df.withColumn('ALSUnit_Int', F.col('ALSUnit').cast(IntegerType()))   # or .cast('int')
fire_cast.select('ALSUnit', 'ALSUnit_Int').show(5)

+-------+-----------+
|ALSUnit|ALSUnit_Int|
+-------+-----------+
|  false|          0|
|   true|          1|
|   true|          1|
|  false|          0|
|  false|          0|
+-------+-----------+
only showing top 5 rows


### 17.5 Using date functions
Now that the dates are real timestamps we can use `F.year()`, `F.month()`, `F.dayofmonth()`, ...

In [61]:
(fire_ts_df.select(F.year('IncidentDate').alias('IncidentYear'))
           .distinct()
           .orderBy('IncidentYear')
           .show())

+------------+
|IncidentYear|
+------------+
|        2000|
|        2001|
|        2002|
|        2003|
|        2004|
|        2005|
|        2006|
|        2007|
|        2008|
|        2009|
|        2010|
|        2011|
|        2012|
|        2013|
|        2014|
|        2015|
|        2016|
|        2017|
|        2018|
+------------+



### 17.6 Aggregation questions
**Q1. What were the most common types of fire calls?**

In [62]:
(fire_ts_df.select('CallType')
           .where(F.col('CallType').isNotNull())
           .groupBy('CallType')
           .count()
           .orderBy('count', ascending=False)
           .show(10, truncate=False))

+-------------------------------+------+
|CallType                       |count |
+-------------------------------+------+
|Medical Incident               |113794|
|Structure Fire                 |23319 |
|Alarms                         |19406 |
|Traffic Collision              |7013  |
|Citizen Assist / Service Call  |2524  |
|Other                          |2166  |
|Outside Fire                   |2094  |
|Vehicle Fire                   |854   |
|Gas Leak (Natural and LP Gases)|764   |
|Water Rescue                   |755   |
+-------------------------------+------+
only showing top 10 rows


In [63]:
# Same question with agg() and a nicer column name
(fire_ts_df.where(F.col('CallType').isNotNull())
           .groupBy('CallType')
           .agg(F.count('CallType').alias('CallTypeCount'))
           .orderBy(F.col('CallTypeCount').desc())
           .show(10, truncate=False))

+-------------------------------+-------------+
|CallType                       |CallTypeCount|
+-------------------------------+-------------+
|Medical Incident               |113794       |
|Structure Fire                 |23319        |
|Alarms                         |19406        |
|Traffic Collision              |7013         |
|Citizen Assist / Service Call  |2524         |
|Other                          |2166         |
|Outside Fire                   |2094         |
|Vehicle Fire                   |854          |
|Gas Leak (Natural and LP Gases)|764          |
|Water Rescue                   |755          |
+-------------------------------+-------------+
only showing top 10 rows


**Q2. Which zip codes accounted for the most calls?**

In [64]:
(fire_ts_df.select('Zipcode')
           .where(F.col('Zipcode').isNotNull())
           .groupBy('Zipcode')
           .count()
           .orderBy('count', ascending=False)
           .show(10))

+-------+-----+
|Zipcode|count|
+-------+-----+
|  94102|21840|
|  94103|20897|
|  94110|14801|
|  94109|14686|
|  94124| 9236|
|  94112| 8421|
|  94115| 7812|
|  94107| 6941|
|  94122| 6355|
|  94133| 6246|
+-------+-----+
only showing top 10 rows


**Q3. Sum of alarms, and the average / minimum / maximum response time over all calls**

In [65]:
(fire_ts_df.select(F.sum('NumAlarms').alias('Total Alarms'),
                   F.avg('ResponseDelayedMins').alias('Avg Response Mins'),
                   F.min('ResponseDelayedMins').alias('Min Response Mins'),
                   F.max('ResponseDelayedMins').alias('Max Response Mins'))
           .show())

+------------+------------------+-----------------+-----------------+
|Total Alarms| Avg Response Mins|Min Response Mins|Max Response Mins|
+------------+------------------+-----------------+-----------------+
|      176170|3.8923641541750342|      0.016666668|          1844.55|
+------------+------------------+-----------------+-----------------+



**Q4. Average and maximum delay for each call type**

In [66]:
(fire_ts_df.groupBy('CallType')
           .agg(F.round(F.avg('ResponseDelayedMins'), 2).alias('AvgDelay'),
                F.max('ResponseDelayedMins').alias('MaxDelay'))
           .orderBy(F.desc('AvgDelay'))
           .show(10, truncate=False))

+-----------------------------------+--------+---------+
|CallType                           |AvgDelay|MaxDelay |
+-----------------------------------+--------+---------+
|Mutual Aid / Assist Outside Agency |38.42   |302.88333|
|Assist Police                      |26.98   |628.61664|
|Train / Rail Incident              |16.45   |406.63333|
|Administrative                     |12.26   |31.983334|
|HazMat                             |7.53    |84.76667 |
|Marine Fire                        |6.93    |13.183333|
|Confined Space / Structure Collapse|6.92    |25.133333|
|Watercraft in Distress             |6.89    |16.333334|
|Suspicious Package                 |6.58    |14.483334|
|High Angle Rescue                  |6.05    |25.616667|
+-----------------------------------+--------+---------+
only showing top 10 rows


**Q5. Calls per year after 2000 (newest first)**

In [67]:
(fire_ts_df.select(F.year('IncidentDate').alias('CallYear'))
           .where('CallYear > 2000')
           .groupBy('CallYear')
           .count()
           .orderBy(F.col('CallYear').desc())
           .show())

fire_df.unpersist()     # done with the case study - free the cache

+--------+-----+
|CallYear|count|
+--------+-----+
|    2018|10136|
|    2017|12135|
|    2016|11609|
|    2015|11458|
|    2014|10775|
|    2013|10020|
|    2012| 9674|
|    2011| 9735|
|    2010| 9341|
|    2009| 8789|
|    2008| 8869|
|    2007| 8255|
|    2006| 8174|
|    2005| 8282|
|    2004| 8283|
|    2003| 8499|
|    2002| 8090|
|    2001| 7713|
+--------+-----+



DataFrame[CallNumber: int, UnitID: string, IncidentNumber: int, CallType: string, CallDate: string, WatchDate: string, CallFinalDisposition: string, AvailableDtTm: string, Address: string, City: string, Zipcode: int, Battalion: string, StationArea: string, Box: int, OriginalPriority: string, Priority: string, FinalPriority: int, ALSUnit: boolean, CallTypeGroup: string, NumAlarms: int, UnitType: string, UnitSequenceInCallDispatch: int, FirePreventionDistrict: string, SupervisorDistrict: string, Neighborhood: string, Location: string, RowID: string, Delay: double]

> For more advanced statistics, look at `df.stat` methods such as `corr()`, `cov()`, `approxQuantile()`, `freqItems()`, `sampleBy()`.

---
## 18. Structured Streaming — DataFrames on data that never stops

### 18.1 Overview
- **Structured Streaming** is a scalable, fault-tolerant stream-processing engine **built on the Spark SQL engine**.
- You write a streaming computation **the same way as a batch computation** on a static DataFrame. Spark runs it incrementally and keeps the result up to date as new data arrives.
- Fault tolerance with **exactly-once** guarantees comes from **checkpointing** and **write-ahead logs**.
- By default it uses a **micro-batch** engine (small batch jobs, latency ~100 ms). Since Spark 2.3 there is also an experimental **continuous** mode (~1 ms latency, *at-least-once*).

### 18.2 Delivery guarantees
| Guarantee | Meaning | Cost |
|-----------|---------|------|
| **at-most-once** | each message is delivered 0 or 1 times — messages may be **lost** | cheapest (fire-and-forget) |
| **at-least-once** | retried until delivered — messages may be **duplicated**, never lost | needs retries + acknowledgements |
| **exactly-once** | delivered exactly once — never lost, never duplicated | most expensive (also needs de-duplication state) |

### 18.3 The programming model: a stream is an unbounded table
- Every new record arriving on the stream is like a **new row appended** to an ever-growing *input table*.
- You write a query on this input table **as if it were static**. Spark turns it into an incremental plan (**incrementalization**): it figures out what *state* to keep so it can update the result each time data arrives.
- Spark does **not** keep all the input; it keeps only the minimal state (e.g. running counts).
- A **trigger** decides *when* to check for new data, and an **output mode** decides *what* is written to the **sink**:

| Output mode | What is written each trigger | Typical use |
|-------------|-----------------------------|-------------|
| **append** (default) | only the **new** rows added to the result | queries without aggregation (rows never change) |
| **complete** | the **entire** result table | aggregations (e.g. running word count) |
| **update** | only the rows that **changed** since the last trigger | aggregations, sinks that can update in place (console, MySQL) |

### 18.4 Five steps to define a streaming query
1. **Define the input source** — `spark.readStream` (a `DataStreamReader`, similar to `DataFrameReader`).
2. **Transform the data** — normal DataFrame operations.
3. **Define the output sink and output mode** — `df.writeStream.format(...).outputMode(...)`.
4. **Specify processing details** — trigger interval and checkpoint location.
5. **Start the query** — `.start()` returns a `StreamingQuery`.

### Before running the next cells: start a text server
We read lines of text from a TCP socket. From the Jupyter home page open **New → Terminal** (it runs on the same machine as this notebook's driver) and start **netcat** listening on port 12345:
```
nc -lk 12345
```
Keep that terminal open and type a few lines of words into it while the query runs.
The **socket source is for testing only** — it gives no fault-tolerance guarantees. Real pipelines read from Kafka or from files (Parquet, JSON, CSV, ...).

In [83]:
# Step 1 - input source. This does NOT read anything yet; it only describes the source.
lines = (spark.readStream.format('socket')
         .option('host', 'localhost')
         .option('port', 12345)
         .load())

print(lines.isStreaming)     # True: this DataFrame has a streaming source
lines.printSchema()          # one string column called "value", one row per line of text

True
root
 |-- value: string (nullable = true)



In [84]:
# Step 2 - transform: exactly the same code you would write for a batch DataFrame

# An aggregation shuffles its data into spark.sql.shuffle.partitions partitions (default 200).
# For a tiny demo stream that means 200 tasks per micro-batch - far too slow. Use a few instead.
spark.conf.set("spark.sql.shuffle.partitions", 4)

words = lines.select(F.explode(F.split(F.col('value'), ' ')).alias('word'))   # split a line into words, one row per word
counts = words.groupBy('word').count()                                        # running count per word

print(words.isStreaming, counts.isStreaming)

True True


`F.split` turns `"hello big world"` into the array `["hello", "big", "world"]`; `F.explode` turns that array into **3 rows**.

### Stateless vs stateful transformations
- **Stateless** (`select`, `filter`, `split`, `explode`, ...): each row is processed on its own. Work on batch and streaming DataFrames.
- **Stateful** (`groupBy().count()`, joins, aggregations): need to remember data across rows / batches (the *state*). Most are supported, a few are not.

### Not supported on streaming DataFrames
- Chains of multiple streaming aggregations, `limit` / `take(n)`, `distinct`.
- Sorting — only after an aggregation **and** in complete mode.
- Some kinds of outer joins.
- Actions that return a result immediately make no sense on an infinite stream:
  - `count()` → use `groupBy().count()` (a running count),
  - `show()` → use the **console sink**,
  - `foreach()` → use `writeStream.foreach(...)`.

In [85]:
# Steps 3 and 4 - sink, output mode, trigger and checkpoint
writer = (counts.writeStream
          .format('console')                          # print each result to the console / Jupyter log
          .outputMode('complete')                     # aggregation -> print the full result table
          .trigger(processingTime='5 seconds')        # check for new data every 5 seconds
          .option('checkpointLocation', f"{OUT_DIR}/chkpnt_wordcount"))

**Triggers** — when to process new data:

| Trigger | Code | Behaviour |
|---------|------|-----------|
| Default | *(none)* | next micro-batch starts as soon as the previous one finishes |
| Processing time | `.trigger(processingTime='5 seconds')` | one micro-batch every fixed interval; skipped if there is no new data; if a batch takes longer than the interval, the next starts immediately |
| Once *(deprecated)* | `.trigger(once=True)` | process everything available in **one** batch, then stop |
| Available now | `.trigger(availableNow=True)` | process everything available in **several** batches, then stop (replacement for *once*) |
| Continuous *(experimental)* | `.trigger(continuous='1 second')` | process continuously, millisecond latency, limited operations |

**Checkpoint location** — a folder (HDFS-compatible) where the query saves its progress. After a failure, the query restarts exactly where it stopped. Required for exactly-once guarantees. If you don't set one, Spark creates a temporary folder.

> **Choosing the interval:** the system is stable only if each batch is processed faster than the batch interval. Start with a conservative interval (5–10 s) and a low data rate, watch the processing time per batch, then tune.

In [86]:
# Step 5 - start the query. start() is NON-blocking: it returns immediately and runs in the background.
query = writer.start()
query.status

{'message': 'Initializing sources',
 'isDataAvailable': False,
 'isTriggerActive': False}

In [87]:
# Type some words into the netcat terminal, then wait here up to 30 s while batches are printed.
# awaitTermination() blocks the cell; without a timeout it waits until the query stops or fails.
query.awaitTermination(30)

-------------------------------------------
Batch: 0
-------------------------------------------
+----+-----+
|word|count|
+----+-----+
+----+-----+



-------------------------------------------
Batch: 1
-------------------------------------------
+-----+-----+
| word|count|
+-----+-----+
|hello|  126|
|   is|  252|
|spark|  252|
|world|  126|
| fast|  126|
|  fun|  126|
+-----+-----+



-------------------------------------------
Batch: 2
-------------------------------------------
+-----+-----+
| word|count|
+-----+-----+
|hello|  130|
|spark|  260|
|   is|  260|
|world|  130|
| fast|  130|
|  fun|  130|
+-----+-----+

-------------------------------------------
Batch: 3
-------------------------------------------
+-----+-----+
| word|count|
+-----+-----+
|hello|  135|
|spark|  270|
|   is|  270|
|world|  135|
| fast|  135|
|  fun|  135|
+-----+-----+



-------------------------------------------
Batch: 4
-------------------------------------------
+-----+-----+
| word|count|
+-----+-----+
|hello|  140|
|spark|  280|
|   is|  280|
|world|  140|
| fast|  140|
|  fun|  140|
+-----+-----+



-------------------------------------------
Batch: 5
-------------------------------------------
+-----+-----+
| word|count|
+-----+-----+
|hello|  145|
|spark|  290|
|   is|  290|
|world|  145|
| fast|  145|
|  fun|  145|
+-----+-----+



-------------------------------------------
Batch: 6
-------------------------------------------
+-----+-----+
| word|count|
+-----+-----+
|hello|  150|
|spark|  300|
|   is|  300|
|world|  150|
| fast|  150|
|  fun|  150|
+-----+-----+



False

In [88]:
query.stop()          # always stop a streaming query when you are done
query.status

{'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}

> **Try it:** change `.outputMode('complete')` to `.outputMode('update')` and re-run steps 3–5 — only the words whose count changed in the last batch are printed. (Use a new checkpoint folder when you change the query.)
>
> A streaming query runs in a **background (daemon) thread**. In a standalone Python script you must call `awaitTermination()` at the end, otherwise the main program exits and kills the query.

---
## 19. 📋 Cheat sheet — summary of everything

### The big ideas
| Idea | Remember |
|---|---|
| **DataFrame** | A distributed table: named columns + a **schema**, split into **partitions** |
| **Immutable** | Every operation returns a **new** DataFrame — the original never changes |
| **Lazy** | Transformations (`select`, `where`, `groupBy`, ...) only build a plan; **actions** (`show`, `count`, `collect`, `write`) run it |
| **Narrow vs wide** | Narrow = no data movement (`select`, `filter`). Wide = **shuffle** (`groupBy`, `sort`, `join`, `distinct`) |
| **Catalyst** | DataFrame code says **what** you want, so Spark can optimize it — RDD lambdas say **how**, so it can't |
| **SparkSession** | One per JVM: `SparkSession.builder.appName('x').getOrCreate()`; `spark.sparkContext` for RDDs |

### Everyday DataFrame code
| Task | DataFrame code |
|------|----------------|
| Create from a list | `spark.createDataFrame(data, "name STRING, age INT")` |
| Read CSV | `spark.read.csv(path, header=True, schema=s)` |
| Read JSON / Parquet | `spark.read.json(path)` / `spark.read.parquet(path)` |
| Look at data | `df.show(n, truncate=False)`, `df.printSchema()`, `df.summary().show()` |
| Column names / types | `df.columns`, `df.dtypes`, `df.schema` |
| Choose columns | `df.select('a', F.col('b') * 2)` |
| SQL expression | `F.expr('b * 2 AS b2')` |
| Rename a result | `(F.col('b') * 2).alias('b2')` |
| Filter rows | `df.where(F.col('a') > 5)` or `df.where("a > 5")` |
| Null checks | `F.col('a').isNull()`, `F.col('a').isNotNull()` |
| New / replace column | `df.withColumn('c', F.col('a') + F.col('b'))` |
| Join strings | `F.concat_ws(' ', 'First', 'Last')` |
| Rename / drop | `df.withColumnRenamed('a', 'x')`, `df.drop('a')` |
| Change type | `F.col('a').cast('int')`, `F.to_timestamp('d', 'MM/dd/yyyy')` |
| Date parts | `F.year('d')`, `F.month('d')`, `F.dayofmonth('d')` |
| Distinct values | `df.select('a').distinct()`, `F.countDistinct('a')` |
| Sort | `df.orderBy(F.desc('a'))`, `df.sort('a', ascending=False)` |
| Group & aggregate | `df.groupBy('a').agg(F.avg('b').alias('avg_b'))` |
| Aggregate everything | `df.agg(F.min('a'), F.max('a'))` |
| If / else column | `F.when(cond, v1).when(cond2, v2).otherwise(v3)` |
| Missing values | `df.na.drop(how=, thresh=, subset=)`, `df.na.fill(value, subset=)` |
| Cache | `df.cache()`, `df.unpersist()`, `spark.catalog.clearCache()` |
| Custom function | `F.udf(func, StringType())` |
| Write | `df.write.mode('overwrite').parquet(path)` |
| Stream | `spark.readStream...` → transform → `df.writeStream...start()` |

### Ways to refer to a column
| Style | Example | Use it when |
|---|---|---|
| plain string | `'age'` | you only need the **name** (no math) |
| attribute | `df.age` | quick, but breaks with spaces or names like `count` |
| indexing | `df['age']` | **always works** |
| `F.col()` | `F.col('age')` | you don't have the DataFrame variable at hand |
| `F.expr()` | `F.expr('age + 1')` | you want to write a small SQL expression |

### Schemas
| Code | Meaning |
|---|---|
| `"Id INT NOT NULL, Name STRING, Tags ARRAY<STRING>"` | DDL string — short and readable (use back-quotes for names with spaces) |
| `StructType([StructField('Id', IntegerType(), False), ...])` | The same schema built in code |
| `spark.createDataFrame(data, schema)` | Spark **checks** every value against the schema |
| `verifySchema=False` | Skip the check — bad values silently become `null` |
| `saved = df.schema` | Reuse a schema to skip inference next time |

### Reading files
| Option | Default | Meaning |
|---|---|---|
| `header=True` | `False` | First line = column names (otherwise `_c0, _c1, ...` and the header becomes a data row) |
| `inferSchema=True` | `False` | Guess the types (otherwise **all strings**) — costs an extra pass over the file |
| `schema=s` | — | Give the types yourself — fastest and safest. Still add `header=True` to skip the header line |
| `samplingRatio=0.01` | `1.0` | Infer the types from a sample only |
| `mode='PERMISSIVE'` | ✅ | Bad row → `null` in the bad fields |
| `mode='DROPMALFORMED'` | | Bad rows are dropped |
| `mode='FAILFAST'` | | Stop with an error at the first bad row |
| `multiLine=True` | `False` | JSON objects that span several lines |

Long form: `spark.read.format('csv').option('header', 'true').schema(s).load(path)` — `path` can be a file, a folder, or a wildcard like `csv/*`.

### Writing files — save modes
| Mode | If the folder already exists |
|---|---|
| `error` / `errorifexists` (default) | raise an error |
| `overwrite` | replace it |
| `append` | add the new data |
| `ignore` | do nothing |

`df.write.format('parquet').mode('overwrite').option('compression', 'snappy').save(path)` → writes a **folder** with one `part-...` file per partition + `_SUCCESS`.
**Parquet** = columnar, compressed, schema stored inside → the recommended format.

### Missing data
| Code | Result |
|---|---|
| `df.na.drop()` | Drop rows with **any** null |
| `df.na.drop(how='all')` | Drop rows where **all** values are null |
| `df.na.drop(thresh=2)` | Keep rows with at least 2 non-null values |
| `df.na.drop(subset=['Sales'])` | Look only at some columns |
| `df.na.fill('No Name')` / `df.na.fill(0)` | Fill string columns / numeric columns only |
| `df.na.fill({'Name': 'NoName', 'Sales': 0})` | Different value per column |

### Structured Streaming
| Step | Code |
|---|---|
| 1. Source | `spark.readStream.format('socket').option('host', ...).option('port', ...).load()` |
| 2. Transform | Normal DataFrame code (`select`, `explode`, `groupBy().count()`, ...) |
| 3. Sink + output mode | `df.writeStream.format('console').outputMode('complete')` |
| 4. Trigger + checkpoint | `.trigger(processingTime='5 seconds').option('checkpointLocation', path)` |
| 5. Start / stop | `query = ....start()`, `query.awaitTermination(30)`, `query.stop()` |

| Output mode | Writes | Use for |
|---|---|---|
| `append` (default) | only new rows | queries **without** aggregation |
| `complete` | the whole result table | aggregations |
| `update` | only rows that changed | aggregations |

### ⚠️ Common mistakes to remember
| Mistake | Remember |
|---|---|
| `from pyspark.sql.functions import *` | Replaces Python's `sum`, `min`, `max`, `round` → use `import pyspark.sql.functions as F` |
| `df.select('Hits * 2')` | A string is a column **name** → use `F.expr('Hits * 2')` or `F.col('Hits') * 2` |
| `cond1 & cond2` without parentheses | Always write `(cond1) & (cond2)` |
| `=` vs `==` | `=` inside SQL strings (`"a = 'x'"`), `==` in column expressions |
| `collect()` / `toPandas()` on big data | Brings **everything** to the driver → use `show()` / `take(n)` |
| Reading a big CSV with `inferSchema=True` | Slow extra pass → define a schema (or use `samplingRatio`) |
| Forgetting `header=True` with a schema | The header line is read as a data row |
| `groupBy('a')` alone | Returns `GroupedData` — add `.count()`, `.avg()` or `.agg(...)` |
| `when` conditions in the wrong order | The **first** true condition wins — go from largest to smallest |
| Writing a Python UDF first | UDFs are slow — look for a built-in function (`F.initcap`, ...) first |
| `cache()` and nothing happens | It is lazy — the data is stored at the **next action** |
| Using `count()` / `show()` on a stream | Use `groupBy().count()` and the console sink instead |
| A real password in a notebook | Read it from an environment variable |